# Projeto TraficGenius: Análise Multivariada de Severidade de Acidentes (Versão Advanced)
Este notebook consolida a execução *End-to-End* do projeto de detecção de fatores de risco para acidentes de trânsito em rodovias americanas.

## Arquitetura do Pipeline (Pipeline Architecture)
1. **Fase 1 (EDA):** Amostragem de parquet, Feature Engineering Espacial (K-Means), Imputação via MICE e Detecção de Outliers híbrida (Mahalanobis + Isolation Forest).
2. **Fase 2 (Premissas):** Testes paramétricos e avaliação iterativa de VIF (Variance Inflation Factor).
3. **Fase 3 a 5 (Modelagem):** Balanceamento de classes via SMOTE. Treinamento comparativo de modelos: **XGBoost Classifier** (Tuning via RandomizedSearch), **CNN 1D** (TensorFlow/Keras, se disponível), **Random Forest** e **Regressão Logística**.
4. **Fase 6:** Geração de Explicabilidade de IA via valores SHAP.


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Ignora avisos indesejados e define o tema padrão do Seaborn
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## Fase 1: EDA, Feature Engineering e Detecção de Outliers

In [ ]:
# Executa o script que faz o processamento pesado e salva o parquet limpo
# O prefixo '!' roda o comando direto no terminal do sistema operacional a partir do notebook
!python pipeline_fase1_eda.py

In [ ]:
# Vamos explorar visualmente o resultado da amostragem limpa
project_root = os.getcwd()
folder_path = os.path.join(project_root, "dataset")
file_path = os.path.join(folder_path, "dataset_amostra_limpa_avancado.parquet")

df = pd.read_parquet(file_path)
print(f"Dataset pronto para uso: {df.shape}")
df.head()

### Visualizando os Clusters Espaciais (K-Means)

In [ ]:
# Desenha o gráfico de dispersão geográfico das coordenadas latitude/longitude
# colorido de acordo com as 20 zonas espaciais criadas pelo MiniBatchKMeans
plt.figure(figsize=(10,6))
sns.scatterplot(x='Longitude_Inicial', y='Latitude_Inicial', hue='Cluster_Espacial', palette='tab20', data=df.sample(10000, random_state=42), alpha=0.5, s=15)
plt.title('Zonas Espaciais de Risco (K-Means)')
plt.show()

## Fase 2: Premissas Estatísticas e Multicolinearidade

In [ ]:
# Executando a validação de normalidade, homocedasticidade e verificação do VIF
!python pipeline_fase2_premissas.py

### Análise de Pressupostos Estatísticos e Multicolinearidade
Abaixo, visualizamos os gráficos gerados durante a Fase 2:
1. **Matriz de Correlação de Pearson:** Avaliação de dependências lineares.
2. **Fator de Inflação de Variância (VIF):** Identificação e descarte de colinearidades críticas (VIF > 10).
3. **Curvas de Normalidade (KDE vs Teórica Gaussiana):** Desvios de normalidade nas variáveis climáticas.


In [ ]:
from IPython.display import Image, display
# 1. Matriz de Correlação
display(Image(filename=os.path.join(folder_path, 'matriz_correlacao.png')))
# 2. VIF
display(Image(filename=os.path.join(folder_path, 'vif_multicolinearidade.png')))
# 3. Curvas de Normalidade
display(Image(filename=os.path.join(folder_path, 'distribuicao_normalidade.png')))

## Fases 3 a 5: Modelagem Preditiva Avançada (XGBoost & Deep Learning)

In [ ]:
# Treinamento e avaliação das métricas de machine learning e redes neurais
!python pipeline_fase3a5_modelagem.py

### Avaliação Visual dos Modelos
Abaixo, visualizamos os gráficos gerados para avaliar o desempenho e dinâmica dos classificadores:
1. **Distribuição das Classes via SMOTE:** Comparação da frequência das severidades antes e depois do balanceamento artificial.
2. **Importância de Variáveis (Feature Importance):** Variáveis que mais influenciam a severidade predita pelo XGBoost.
3. **Matriz de Confusão (Heatmap):** Relação percentual e absoluta de acertos e erros do classificador.
4. **Comparativo de Performance dos Modelos:** Gráfico comparativo neon exibindo F1-Score Macro e Acurácia Global de todos os modelos.


In [ ]:
from IPython.display import Image, display
# 1. Distribuição SMOTE
display(Image(filename=os.path.join(folder_path, 'distribuicao_classes_smote.png')))
# 2. Importância de Variáveis
display(Image(filename=os.path.join(folder_path, 'importancia_features_xgboost.png')))
# 3. Matriz de Confusão
display(Image(filename=os.path.join(folder_path, 'matriz_confusao_xgboost.png')))
# 4. Comparativo de Performance dos Modelos
display(Image(filename=os.path.join(folder_path, 'comparativo_performance_modelos.png')))